# Model Comparison: LDA vs. BERTopic

This notebook compares the two topic modeling approaches — LDA and BERTopic — across multiple dimensions: coherence score, topic interpretability, document coverage, and alignment with the known 20 Newsgroup categories.

**Input**: Models saved in `../03_lda/output/` and `../04_bertopic/output/`  
**Output**: Comparison visualizations saved to `./output/`

## Imports

In [ ]:
import boto3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import pickle
import warnings
warnings.filterwarnings('ignore')

from gensim import corpora
from gensim.models import CoherenceModel

print('All imports successful')

## Helper Class

In [ ]:
class ModelComparison:
    """Compare LDA and BERTopic topic models."""
    
    def __init__(self, str_bucket, str_dirname_output):
        self.str_bucket = str_bucket
        self.str_dirname_output = str_dirname_output
        self.s3_client = boto3.client('s3', region_name='us-east-1')
        self.df_data = None
        self.dict_lda = None
        self.dict_bertopic = None
        self.flt_lda_coherence = None
        self.flt_bertopic_coherence = None
        self.set_plot_style()
    
    def set_plot_style(self):
        """Configure matplotlib/seaborn styling."""
        sns.set_style('whitegrid')
        plt.rcParams['figure.figsize'] = (14, 6)
        plt.rcParams['font.size'] = 11
    
    def import_data(self):
        """Load preprocessed data from S3."""
        str_uri = f's3://{self.str_bucket}/02_preprocessing/newsgroups_preprocessed.parquet'
        self.df_data = pd.read_parquet(str_uri)
        print(f'Loaded {len(self.df_data):,} posts from S3')
    
    def load_models(self):
        """Load saved LDA and BERTopic models."""
        with open('../03_lda/output/lda_model.pkl', 'rb') as f:
            self.dict_lda = pickle.load(f)
        print(f'Loaded LDA model ({self.dict_lda["int_best_num_topics"]} topics)')
        
        with open('../04_bertopic/output/bertopic_model.pkl', 'rb') as f:
            self.dict_bertopic = pickle.load(f)
        int_n_bertopic = len(set(self.dict_bertopic['lst_topics'])) - (1 if -1 in self.dict_bertopic['lst_topics'] else 0)
        print(f'Loaded BERTopic model ({int_n_bertopic} topics)')
    
    def compute_coherence_scores(self):
        """Compute coherence scores for both models."""
        lst_texts = [str_doc.split() for str_doc in self.df_data['text_clean']]
        dictionary = corpora.Dictionary(lst_texts)
        dictionary.filter_extremes(no_below=15, no_above=0.5)
        
        # LDA coherence
        lda_model = self.dict_lda['lda_model']
        lda_coherence = CoherenceModel(
            model=lda_model,
            texts=lst_texts,
            dictionary=self.dict_lda['dictionary'],
            coherence='c_v'
        )
        self.flt_lda_coherence = lda_coherence.get_coherence()
        
        # BERTopic coherence
        bertopic_model = self.dict_bertopic['topic_model']
        lst_topic_ids = [int_t for int_t in bertopic_model.get_topic_info()['Topic'] if int_t != -1]
        lst_topic_words = []
        for int_t in lst_topic_ids[:20]:
            lst_words = [str_w for str_w, _ in bertopic_model.get_topic(int_t)[:10]]
            lst_topic_words.append(lst_words)
        
        bertopic_coherence = CoherenceModel(
            topics=lst_topic_words,
            texts=lst_texts,
            dictionary=dictionary,
            coherence='c_v'
        )
        self.flt_bertopic_coherence = bertopic_coherence.get_coherence()
        
        print(f'LDA Coherence (c_v): {self.flt_lda_coherence:.4f}')
        print(f'BERTopic Coherence (c_v): {self.flt_bertopic_coherence:.4f}')
    
    def plot_coherence_comparison(self):
        """Bar chart comparing coherence scores."""
        fig, ax = plt.subplots(figsize=(10, 6))
        
        lst_models = ['LDA', 'BERTopic']
        lst_scores = [self.flt_lda_coherence, self.flt_bertopic_coherence]
        lst_colors = ['steelblue', '#DD8452']
        
        bars = ax.bar(lst_models, lst_scores, color=lst_colors, edgecolor='black', width=0.5)
        ax.set_ylabel('Coherence Score (c_v)', fontsize=12, fontweight='bold')
        ax.set_title('Topic Coherence: LDA vs. BERTopic', fontsize=14, fontweight='bold')
        
        for bar, flt_score in zip(bars, lst_scores):
            ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.005,
                    f'{flt_score:.4f}', ha='center', va='bottom',
                    fontweight='bold', fontsize=13)
        
        ax.set_ylim(0, max(lst_scores) * 1.15)
        
        plt.tight_layout()
        plt.savefig(f'{self.str_dirname_output}/01_coherence_comparison.png',
                    bbox_inches='tight', dpi=150)
        plt.show()
        print('Saved: 01_coherence_comparison.png')
    
    def plot_topic_count_comparison(self):
        """Compare number of topics discovered."""
        int_lda_topics = self.dict_lda['int_best_num_topics']
        lst_bertopic_topics = self.dict_bertopic['lst_topics']
        int_bertopic_topics = len(set(lst_bertopic_topics)) - (1 if -1 in lst_bertopic_topics else 0)
        int_outliers = lst_bertopic_topics.count(-1) if isinstance(lst_bertopic_topics, list) else (np.array(lst_bertopic_topics) == -1).sum()
        
        fig, axes = plt.subplots(1, 2, figsize=(16, 6))
        
        # topic count
        lst_models = ['LDA', 'BERTopic']
        lst_counts = [int_lda_topics, int_bertopic_topics]
        lst_colors = ['steelblue', '#DD8452']
        
        bars = axes[0].bar(lst_models, lst_counts, color=lst_colors, edgecolor='black', width=0.5)
        axes[0].set_ylabel('Number of Topics', fontsize=12, fontweight='bold')
        axes[0].set_title('Topics Discovered', fontsize=14, fontweight='bold')
        axes[0].axhline(y=20, color='red', linestyle='--', linewidth=2, label='True categories (20)')
        axes[0].legend(fontsize=11)
        
        for bar, int_count in zip(bars, lst_counts):
            axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.3,
                         str(int_count), ha='center', va='bottom',
                         fontweight='bold', fontsize=13)
        
        # document coverage
        int_total = len(self.df_data)
        int_lda_covered = int_total  # LDA assigns every document
        int_bertopic_covered = int_total - int_outliers
        
        lst_covered = [int_lda_covered / int_total * 100, int_bertopic_covered / int_total * 100]
        bars = axes[1].bar(lst_models, lst_covered, color=lst_colors, edgecolor='black', width=0.5)
        axes[1].set_ylabel('% Documents Assigned', fontsize=12, fontweight='bold')
        axes[1].set_title('Document Coverage', fontsize=14, fontweight='bold')
        axes[1].set_ylim(0, 110)
        
        for bar, flt_pct in zip(bars, lst_covered):
            axes[1].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 1,
                         f'{flt_pct:.1f}%', ha='center', va='bottom',
                         fontweight='bold', fontsize=13)
        
        plt.tight_layout()
        plt.savefig(f'{self.str_dirname_output}/02_topic_count_comparison.png',
                    bbox_inches='tight', dpi=150)
        plt.show()
        print('Saved: 02_topic_count_comparison.png')
    
    def plot_side_by_side_topics(self):
        """Side-by-side top words comparison for select topics."""
        lda_model = self.dict_lda['lda_model']
        bertopic_model = self.dict_bertopic['topic_model']
        
        # get first 6 topics from each
        int_n_show = min(6, lda_model.num_topics)
        
        fig, axes = plt.subplots(int_n_show, 2, figsize=(16, 3.5 * int_n_show))
        
        # LDA topics (left column)
        for int_i in range(int_n_show):
            lst_topic_words = lda_model.show_topic(int_i, topn=8)
            lst_words = [str_w for str_w, _ in lst_topic_words]
            lst_weights = [flt_w for _, flt_w in lst_topic_words]
            
            axes[int_i, 0].barh(range(len(lst_words)), lst_weights,
                                color='steelblue', edgecolor='black')
            axes[int_i, 0].set_yticks(range(len(lst_words)))
            axes[int_i, 0].set_yticklabels(lst_words)
            axes[int_i, 0].invert_yaxis()
            axes[int_i, 0].set_title(f'LDA Topic {int_i + 1}', fontsize=11, fontweight='bold')
        
        # BERTopic topics (right column)
        lst_bertopic_ids = [int_t for int_t in bertopic_model.get_topic_info()['Topic'] if int_t != -1][:int_n_show]
        
        for int_i, int_topic_id in enumerate(lst_bertopic_ids):
            lst_topic_words = bertopic_model.get_topic(int_topic_id)
            if not lst_topic_words:
                continue
            lst_words = [str_w for str_w, _ in lst_topic_words[:8]]
            lst_scores = [flt_s for _, flt_s in lst_topic_words[:8]]
            
            axes[int_i, 1].barh(range(len(lst_words)), lst_scores,
                                color='#DD8452', edgecolor='black')
            axes[int_i, 1].set_yticks(range(len(lst_words)))
            axes[int_i, 1].set_yticklabels(lst_words)
            axes[int_i, 1].invert_yaxis()
            axes[int_i, 1].set_title(f'BERTopic Topic {int_topic_id}', fontsize=11, fontweight='bold')
        
        plt.suptitle('Side-by-Side Topic Comparison: LDA vs. BERTopic',
                     fontsize=14, fontweight='bold', y=1.01)
        plt.tight_layout()
        plt.savefig(f'{self.str_dirname_output}/03_side_by_side_topics.png',
                    bbox_inches='tight', dpi=150)
        plt.show()
        print('Saved: 03_side_by_side_topics.png')
    
    def plot_summary_table(self):
        """Create a visual summary comparison table."""
        int_lda_topics = self.dict_lda['int_best_num_topics']
        lst_bertopic_topics = self.dict_bertopic['lst_topics']
        int_bertopic_topics = len(set(lst_bertopic_topics)) - (1 if -1 in lst_bertopic_topics else 0)
        int_outliers = lst_bertopic_topics.count(-1) if isinstance(lst_bertopic_topics, list) else (np.array(lst_bertopic_topics) == -1).sum()
        int_total = len(self.df_data)
        
        lst_metrics = ['Coherence (c_v)', 'Topics Discovered', 'Document Coverage',
                       'Approach', 'Speed', 'Interpretability']
        lst_lda_vals = [
            f'{self.flt_lda_coherence:.4f}',
            str(int_lda_topics),
            '100%',
            'Probabilistic',
            'Fast',
            'Word distributions'
        ]
        lst_bertopic_vals = [
            f'{self.flt_bertopic_coherence:.4f}',
            str(int_bertopic_topics),
            f'{(int_total - int_outliers) / int_total * 100:.1f}%',
            'Embedding-based',
            'Slower (embeddings)',
            'c-TF-IDF + clusters'
        ]
        
        fig, ax = plt.subplots(figsize=(14, 5))
        ax.axis('off')
        
        table = ax.table(
            cellText=list(zip(lst_metrics, lst_lda_vals, lst_bertopic_vals)),
            colLabels=['Metric', 'LDA', 'BERTopic'],
            cellLoc='center',
            loc='center'
        )
        table.auto_set_font_size(False)
        table.set_fontsize(12)
        table.scale(1, 2)
        
        # style header
        for int_j in range(3):
            table[0, int_j].set_facecolor('#4C72B0')
            table[0, int_j].set_text_props(color='white', fontweight='bold')
        
        # alternate row colors
        for int_i in range(1, len(lst_metrics) + 1):
            str_color = '#f0f0f0' if int_i % 2 == 0 else 'white'
            for int_j in range(3):
                table[int_i, int_j].set_facecolor(str_color)
        
        ax.set_title('Model Comparison Summary', fontsize=14, fontweight='bold', pad=20)
        
        plt.tight_layout()
        plt.savefig(f'{self.str_dirname_output}/04_summary_table.png',
                    bbox_inches='tight', dpi=150)
        plt.show()
        print('Saved: 04_summary_table.png')
    
    def print_summary(self):
        """Print comparison summary."""
        int_lda_topics = self.dict_lda['int_best_num_topics']
        lst_bertopic_topics = self.dict_bertopic['lst_topics']
        int_bertopic_topics = len(set(lst_bertopic_topics)) - (1 if -1 in lst_bertopic_topics else 0)
        
        print('\n' + '='*60)
        print('COMPARISON SUMMARY')
        print('='*60)
        print(f'\n{"Metric":<25} {"LDA":<20} {"BERTopic":<20}')
        print('-'*60)
        print(f'{"Coherence (c_v)":<25} {self.flt_lda_coherence:<20.4f} {self.flt_bertopic_coherence:<20.4f}')
        print(f'{"Topics Discovered":<25} {int_lda_topics:<20} {int_bertopic_topics:<20}')
        print(f'{"True Categories":<25} {"20":<20} {"20":<20}')
        print('='*60)

## Constants

In [ ]:
# S3 configuration
str_bucket = 'topic-modeling-demo'

# output directory
str_dirname_output = './output'

print(f'S3 Bucket: {str_bucket}')
print(f'Output Directory: {str_dirname_output}')

## Output Directory

In [ ]:
try:
    os.mkdir(str_dirname_output)
    print(f'Created output directory: {str_dirname_output}')
except FileExistsError:
    print(f'Output directory already exists: {str_dirname_output}')

## Load Data and Models

In [ ]:
cls_comparison = ModelComparison(str_bucket=str_bucket, str_dirname_output=str_dirname_output)
cls_comparison.import_data()
cls_comparison.load_models()

## Compute Coherence Scores

In [ ]:
cls_comparison.compute_coherence_scores()

## Visualize Comparison

In [ ]:
cls_comparison.plot_coherence_comparison()

In [ ]:
cls_comparison.plot_topic_count_comparison()

In [ ]:
cls_comparison.plot_side_by_side_topics()

In [ ]:
cls_comparison.plot_summary_table()

In [ ]:
cls_comparison.print_summary()